## Making a figure with high-z double-peaked LAEs (z > 4)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import astropy.table as aptb
from pathlib import Path

In [ ]:
# Load megatable of Lyman alpha sources
# Which spectra source to use? R21 or APER
SPEC_SOURCE = "R21"  #  Where are the spectra from? 'R21' for Richard+21, 'APER' for apertures
SPEC_TYPE   = 'weight_skysub' if SPEC_SOURCE == "R21" else '2fwhm'

tabfile = f"../megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_absv_{SPEC_TYPE}.fits"
tabhdul = fits.open(tabfile)
megatable = aptb.Table(tabhdul[1].data)

# Filter only double peaked sources with z > 4
dp_mask = megatable["SNRB"] > 5.0
z_mask = megatable["z"] > 4.0

dp_megatable = megatable[dp_mask & z_mask]

# Print out the number of double peaked sources at z > 4
print(f"Number of double peaked sources at z > 4: {len(dp_megatable)}")

In [ ]:
# Make a plot of the spectra of these sources, getting the figures already saved in the corresponding plot directory
fig_cols = 4
fig_rows = int(np.ceil(len(dp_megatable) / fig_cols))
fig, axs = plt.subplots(fig_rows, fig_cols, figsize=(20, 3 * fig_rows))
axs = axs.flatten()  # Flatten in case of multiple rows
nax = 0
for idx, row in enumerate(dp_megatable):
    cluster = row['CLUSTER']
    source = row['iden']
    z = row['z']
    # Bring up the saved plot of this source for visual inspection
    plot_dir = Path(f"../../muse_data/{cluster}/plots/{source}/LYALPHA_fit_{SPEC_TYPE}.png")
    if plot_dir.exists():
        img = plt.imread(plot_dir)
        axs[nax].imshow(img)
        # Remove ticks and spines but keep the label area visible
        axs[nax].set_xticks([])
        axs[nax].set_yticks([])
        for spine in axs[nax].spines.values():
            spine.set_visible(False)
        # Cut off the bottom 0.05 of the plot to remove the x-axis labels as they contain an error
        axs[nax].set_ylim(img.shape[0] * 0.95, img.shape[0] * 0.1)
        # Cut off the left 0.05 of the plot to remove the y-axis labels as they contain an error
        axs[nax].set_xlim(img.shape[1] * 0.08, img.shape[1])
        # Replace x-axis labels
        axs[nax].set_xlabel(r"Wavelength [\AA]", fontsize=10)
        # Replace y-axis labels
        axs[nax].set_ylabel(r"Flux density [10$^{-20}$ erg s$^{-1}$ cm$^{-2}$ \AA$^{-1}$]", fontsize=10)
        # Replace title
        axs[nax].set_title(f"{cluster} - {source} at $z={z:.2f}$", fontsize=12)
        nax += 1
    else:
        print(f"  Plot not found at {plot_dir}")


plt.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01, wspace=0, hspace=-0.1)
plt.savefig(f"../plots/highz_dp_profiles.pdf", dpi=200,
            bbox_inches='tight')
plt.show()
